# 🏍️ Training YOLOv12: Deteksi Perlengkapan Berkendara Motor (100 Epochs - Ringan & Cepat)
Notebook ini dikonfigurasi ulang untuk melatih model YOLOv12 menggunakan dataset baru Anda yang sudah dibersihkan secara cepat dan efisien (Maksimal 100 Epoch).

### ⚠️ PENTING: AKTIFKAN GPU SEBELUM MEMULAI
Sebelum menjalankan sel kode di bawah ini, pastikan Anda menggunakan runtime GPU:
1. Klik menu **Runtime** di bagian atas Colab.
2. Pilih **Change runtime type**.
3. Pada bagian *Hardware accelerator*, pilih **T4 GPU**.
4. Klik **Save**.

## Langkah 1: Instalasi Library Yang Dibutuhkan
Kita memerlukan pustaka `ultralytics` untuk menggunakan YOLOv12 dan `roboflow` untuk mengunduh dataset.

In [ ]:
!pip install ultralytics roboflow

## Langkah 2: Mengunduh Dataset secara Otomatis
Sel kode di bawah ini akan membersihkan sisa file zip yang rusak, mencari versi terbaru project **`baru-lagi`** secara otomatis di server Roboflow, lalu mengunduhnya secara bersih.

In [ ]:
import os
import shutil
from roboflow import Roboflow

# 1. Bersihkan sisa folder/zip yang rusak agar tidak mengganggu
print("Membersihkan folder lama...")
for name in os.listdir("/content"):
    if name.startswith("baru-lagi"):
        full_path = os.path.join("/content", name)
        if os.path.isdir(full_path):
            shutil.rmtree(full_path)
            print(f"Folder dihapus: {full_path}")
        elif os.path.isfile(full_path):
            os.remove(full_path)
            print(f"Berkas dihapus: {full_path}")

# 2. Inisialisasi Roboflow API
rf = Roboflow(api_key="GcjsgvWWLCDmts88LhHy")
project_name = "baru-lagi"

print(f"Menghubungkan ke project: {project_name}...")
project = rf.workspace("cicangoding").project(project_name)

# 3. Deteksi versi aktif terbaru secara otomatis
available_versions = [v.version for v in project.versions()]
if not available_versions:
    raise RuntimeError("Tidak ada versi aktif yang ditemukan pada project ini.")

latest_version = max(available_versions)
print(f"Daftar versi aktif di Roboflow: {available_versions}")
print(f"Menggunakan versi terbaru: {latest_version}")

# 4. Unduh dataset format yolov8 (100% kompatibel dengan yolov12)
version = project.version(latest_version)
print(f"Memulai download dataset...")
dataset = version.download("yolov8")

data_yaml_path = os.path.join(dataset.location, "data.yaml")
print(f"\n[SUKSES] Dataset berhasil diunduh ke: {dataset.location}")
print(f"Path konfigurasi YAML: {data_yaml_path}")

## Langkah 3: Melatih Model YOLOv12 (Latih Ringan - 100 Epochs)
Konfigurasi yang disesuaikan agar proses training cepat dan ringan namun tetap akurat:
- **Model**: YOLOv12s (Small) - Memiliki performa yang jauh lebih baik daripada Nano, namun sangat cepat dilatih di T4 GPU Colab.
- **imgsz=640**: Ukuran gambar standar YOLO yang sangat cepat diproses oleh GPU dan bebas resiko memory limit (OOM).
- **batch=16**: Ukuran batch stabil untuk mempercepat pembelajaran.
- **epochs=100**: Batasan maksimal agar training tidak memakan waktu lama.
- **optimizer="AdamW"**: Pembelajaran yang stabil dengan learning rate yang disesuaikan (`lr0=0.001`).
- **freeze=10**: Membekukan 10 layer pertama (Backbone) agar model tidak overfit dan training berjalan lebih cepat.

In [ ]:
import torch
from ultralytics import YOLO

# Pastikan GPU terdeteksi aktif
device_to_use = 0 if torch.cuda.is_available() else "cpu"
print(f"Menggunakan device: {device_to_use} (CUDA tersedia: {torch.cuda.is_available()})")
if device_to_use == "cpu":
    print("[PERINGATAN] GPU T4 Colab belum diaktifkan! Harap ubah runtime ke GPU agar training cepat selesai.")

# Inisialisasi model YOLOv12s (Small)
model = YOLO("yolo12s.pt")

# Jalankan proses training
results = model.train(
    data=data_yaml_path,
    epochs=100,             # Maksimal 100 epoch agar cepat dan tidak berat
    imgsz=640,              # Resolusi 640 standar (ringan dan cepat)
    batch=16,               # Batch size 16 agar training cepat selesai
    device=device_to_use,   # Menggunakan GPU Colab
    
    # Pengaturan Akurasi Maksimal:
    optimizer="AdamW",      # Menggunakan AdamW yang stabil
    lr0=0.001,              # Diturunkan ke 0.001 agar AdamW stabil dan tidak merusak bobot pretrained
    lrf=0.01,
    cos_lr=True,            
    patience=20,            # Stop jika mAP tidak naik dalam 20 epoch
    freeze=10,              # Membekukan 10 layer pertama agar stabil pada data sedikit
    
    val=True                # Menghasilkan grafik validasi mAP
)

## Langkah 4: Mengunduh File Model Terbaik (`best.pt`)
Jalankan kode di bawah ini untuk mengunduh model hasil latihan terbaik ke komputer lokal Anda untuk diintegrasikan dengan web dashboard Streamlit.

In [ ]:
from google.colab import files
import os

best_model_path = '/content/runs/detect/train/weights/best.pt'
if os.path.exists(best_model_path):
    print("Mengunduh file best.pt ke komputer Anda...")
    files.download(best_model_path)
else:
    print("[ERROR] File model best.pt belum terbentuk. Pastikan proses training di Langkah 3 selesai sepenuhnya!")